In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_disease_dataset.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["village_id", "date"]).reset_index(drop=True)
df.head()

,village_id,village_name,latitude,longitude,population,water_source_type,date,ph_level,turbidity_ntu,residual_chlorine_mg_l,bacterial_contamination,rainfall_mm,temperature_c,sanitation_index,historical_cases_last_4_weeks,disease_type,case_count,outbreak_risk_level
0,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-07,6.84,1.95,0.456,0,13.1,32.4,0.34,0,diarrhea,8,High
1,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-14,7.33,2.31,0.462,0,6.4,25.7,0.34,8,typhoid,18,High
2,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-21,5.94,2.49,0.326,0,5.4,28.9,0.34,18,diarrhea,6,Medium
3,V001,Village_1,16.57454,74.31185,10470,pond,2024-01-28,6.58,1.78,0.382,0,8.6,28.0,0.34,6,cholera,2,Low
4,V001,Village_1,16.57454,74.31185,10470,pond,2024-02-04,6.99,2.84,0.274,0,7.0,37.4,0.34,2,hepatitis_a,6,Medium


In [2]:
df["cases_lag_1"] = df.groupby("village_id")["case_count"].shift(1)
df["cases_lag_2"] = df.groupby("village_id")["case_count"].shift(2)
df["cases_lag_3"] = df.groupby("village_id")["case_count"].shift(3)

df[["village_id", "date", "case_count", "cases_lag_1", "cases_lag_2", "cases_lag_3"]].head(10)

,village_id,date,case_count,cases_lag_1,cases_lag_2,cases_lag_3
0,V001,2024-01-07,8,NaN,NaN,NaN
1,V001,2024-01-14,18,8.0,NaN,NaN
2,V001,2024-01-21,6,18.0,8.0,NaN
3,V001,2024-01-28,2,6.0,18.0,8.0
4,V001,2024-02-04,6,2.0,6.0,18.0
5,V001,2024-02-11,2,6.0,2.0,6.0
6,V001,2024-02-18,12,2.0,6.0,2.0
7,V001,2024-02-25,14,12.0,2.0,6.0
8,V001,2024-03-03,6,14.0,12.0,2.0
9,V001,2024-03-10,10,6.0,14.0,12.0


In [3]:
df["future_risk_level"] = df.groupby("village_id")["outbreak_risk_level"].shift(-1)

df[["village_id", "date", "outbreak_risk_level", "future_risk_level"]].head(10)

,village_id,date,outbreak_risk_level,future_risk_level
0,V001,2024-01-07,High,High
1,V001,2024-01-14,High,Medium
2,V001,2024-01-21,Medium,Low
3,V001,2024-01-28,Low,Medium
4,V001,2024-02-04,Medium,Low
5,V001,2024-02-11,Low,High
6,V001,2024-02-18,High,High
7,V001,2024-02-25,High,Medium
8,V001,2024-03-03,Medium,High
9,V001,2024-03-10,High,High


In [4]:
print("Before dropping:", df.shape)
df_forecast = df.dropna(subset=["cases_lag_1", "cases_lag_2", "cases_lag_3", "future_risk_level"])
print("After dropping:", df_forecast.shape)

Before dropping: (1040, 22)
After dropping: (960, 22)


In [5]:
df_forecast_encoded = pd.get_dummies(df_forecast, columns=["water_source_type", "disease_type"], drop_first=True, dtype=int)

risk_mapping = {"Low": 0, "Medium": 1, "High": 2}
df_forecast_encoded["future_risk_encoded"] = df_forecast_encoded["future_risk_level"].map(risk_mapping)

df_forecast_encoded.shape

(960, 27)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

drop_cols = ["village_id", "village_name", "date", "outbreak_risk_level",
             "future_risk_level", "future_risk_encoded", "case_count"]

X_forecast = df_forecast_encoded.drop(columns=drop_cols)
y_forecast = df_forecast_encoded["future_risk_encoded"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_forecast, y_forecast, test_size=0.2, random_state=42, stratify=y_forecast
)

forecast_model = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
forecast_model.fit(X_train_f, y_train_f)

print("Forecast Model Train Accuracy:", round(forecast_model.score(X_train_f, y_train_f), 3))
print("Forecast Model Test Accuracy:", round(forecast_model.score(X_test_f, y_test_f), 3))

Forecast Model Train Accuracy: 0.919
Forecast Model Test Accuracy: 0.677


In [7]:
naive_prediction = df_forecast_encoded["outbreak_risk_level"].map(risk_mapping)  # "next week = same as this week"
naive_accuracy = (naive_prediction == df_forecast_encoded["future_risk_encoded"]).mean()

print("Naive baseline accuracy (assume next week = this week):", round(naive_accuracy, 3))
print("Our forecast model test accuracy:", round(forecast_model.score(X_test_f, y_test_f), 3))

Naive baseline accuracy (assume next week = this week): 0.603
Our forecast model test accuracy: 0.677


GIS START

In [8]:
# import sys
# !{sys.executable} -m pip install folium

In [9]:
import pandas as pd
import joblib

df = pd.read_csv("../data/processed/features_disease_dataset.csv")
rf_model = joblib.load("../models/random_forest_model.pkl")

df.head()

,village_id,village_name,latitude,longitude,population,date,ph_level,turbidity_ntu,residual_chlorine_mg_l,bacterial_contamination,...,sanitation_gap,week_number,month,water_source_type_handpump,water_source_type_pond,water_source_type_river,disease_type_diarrhea,disease_type_hepatitis_a,disease_type_typhoid,outbreak_risk_level_encoded
0,V001,Village_1,16.57454,74.31185,10470,2024-01-07,6.84,1.95,0.456,0,...,0.66,1,1,False,True,False,True,False,False,2
1,V001,Village_1,16.57454,74.31185,10470,2024-01-14,7.33,2.31,0.462,0,...,0.66,2,1,False,True,False,False,False,True,2
2,V001,Village_1,16.57454,74.31185,10470,2024-01-21,5.94,2.49,0.326,0,...,0.66,3,1,False,True,False,True,False,False,1
3,V001,Village_1,16.57454,74.31185,10470,2024-01-28,6.58,1.78,0.382,0,...,0.66,4,1,False,True,False,False,False,False,0
4,V001,Village_1,16.57454,74.31185,10470,2024-02-04,6.99,2.84,0.274,0,...,0.66,5,2,False,True,False,False,True,False,1


In [10]:
df["date"] = pd.to_datetime(df["date"])
latest_data = df.sort_values("date").groupby("village_id").tail(1).reset_index(drop=True)

print(latest_data.shape)
latest_data[["village_id", "village_name", "date", "latitude", "longitude"]]

(20, 28)


,village_id,village_name,date,latitude,longitude
0,V010,Village_10,2024-12-29,16.90807,73.74645
1,V018,Village_18,2024-12-29,16.72476,73.79767
2,V001,Village_1,2024-12-29,16.57454,74.31185
3,V002,Village_2,2024-12-29,17.15071,73.83949
4,V003,Village_3,2024-12-29,16.93199,73.99214
5,V004,Village_4,2024-12-29,16.79866,74.06636
6,V005,Village_5,2024-12-29,16.35602,74.15607
7,V006,Village_6,2024-12-29,16.35599,74.48518
8,V007,Village_7,2024-12-29,16.25808,73.89967
9,V019,Village_19,2024-12-29,16.63195,74.38423


In [11]:
drop_columns = ["village_id", "village_name", "date", "outbreak_risk_level",
                 "outbreak_risk_level_encoded", "case_count"]

X_latest = latest_data.drop(columns=drop_columns)
predictions = rf_model.predict(X_latest)

risk_mapping_reverse = {0: "Low", 1: "Medium", 2: "High"}
latest_data["predicted_risk"] = pd.Series(predictions).map(risk_mapping_reverse)

latest_data[["village_name", "predicted_risk"]]

,village_name,predicted_risk
0,Village_10,Medium
1,Village_18,Low
2,Village_1,High
3,Village_2,High
4,Village_3,Low
5,Village_4,Medium
6,Village_5,Low
7,Village_6,High
8,Village_7,Medium
9,Village_19,Medium


In [20]:
import folium

center_lat = latest_data["latitude"].mean()
center_lon = latest_data["longitude"].mean()

risk_map = folium.Map(location=[center_lat, center_lon], zoom_start=10)
risk_map

In [21]:
color_mapping = {"Low": "green", "Medium": "orange", "High": "red"}

for index in range(len(latest_data)):
    row = latest_data.iloc[index]
    popup_text = f"{row['village_name']}<br>Risk: {row['predicted_risk']}<br>Population: {row['population']}"

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=10,
        popup=popup_text,
        color=color_mapping[row["predicted_risk"]],
        fill=True,
        fill_color=color_mapping[row["predicted_risk"]],
        fill_opacity=0.8
    ).add_to(risk_map)

risk_map

In [23]:
risk_map.save("village_risk_map.html")
print("Map saved as village_risk_map.html — open this file in your browser.")

Map saved as village_risk_map.html — open this file in your browser.


In [24]:
import numpy as np

conditions = [
    (latest_data["predicted_risk"] == "High") & (latest_data["residual_chlorine_mg_l"] < 0.2),
    (latest_data["predicted_risk"] == "High") & (latest_data["turbidity_ntu"] > 5),
    (latest_data["predicted_risk"] == "High") & (latest_data["bacterial_contamination"] == 1),
    (latest_data["predicted_risk"] == "Medium") & (latest_data["sanitation_gap"] > 0.5),
    (latest_data["predicted_risk"] == "Low"),
]

choices = [
    "URGENT: Chlorinate water supply immediately; distribute water purification tablets.",
    "URGENT: Investigate turbidity source (runoff/sediment); recommend boiling water before use.",
    "URGENT: Bacterial contamination detected; issue boil-water advisory and test supply.",
    "Improve sanitation infrastructure; monitor closely over next 2 weeks.",
    "No immediate action needed; continue routine monitoring.",
]

latest_data["recommendation"] = np.select(conditions, choices, default="Risk detected; conduct manual water quality inspection.")

latest_data[["village_name", "predicted_risk", "recommendation"]]

,village_name,predicted_risk,recommendation
0,Village_10,Medium,Risk detected; conduct manual water quality in...
1,Village_18,Low,No immediate action needed; continue routine m...
2,Village_1,High,Risk detected; conduct manual water quality in...
3,Village_2,High,Risk detected; conduct manual water quality in...
4,Village_3,Low,No immediate action needed; continue routine m...
5,Village_4,Medium,Risk detected; conduct manual water quality in...
6,Village_5,Low,No immediate action needed; continue routine m...
7,Village_6,High,Risk detected; conduct manual water quality in...
8,Village_7,Medium,Improve sanitation infrastructure; monitor clo...
9,Village_19,Medium,Risk detected; conduct manual water quality in...


In [25]:
water_quality_notes = [
    (latest_data["residual_chlorine_mg_l"] < 0.2),
    (latest_data["turbidity_ntu"] > 5),
]

note_choices = [
    "Chlorine level below WHO-recommended minimum (0.2 mg/L).",
    "Turbidity above acceptable threshold (5 NTU).",
]

latest_data["water_quality_note"] = np.select(water_quality_notes, note_choices, default="Water quality parameters within acceptable range.")

latest_data[["village_name", "residual_chlorine_mg_l", "turbidity_ntu", "water_quality_note"]]

,village_name,residual_chlorine_mg_l,turbidity_ntu,water_quality_note
0,Village_10,0.282,2.29,Water quality parameters within acceptable range.
1,Village_18,0.379,1.53,Water quality parameters within acceptable range.
2,Village_1,0.437,3.45,Water quality parameters within acceptable range.
3,Village_2,0.317,2.81,Water quality parameters within acceptable range.
4,Village_3,0.288,0.93,Water quality parameters within acceptable range.
5,Village_4,0.261,3.19,Water quality parameters within acceptable range.
6,Village_5,0.469,0.41,Water quality parameters within acceptable range.
7,Village_6,0.222,1.23,Water quality parameters within acceptable range.
8,Village_7,0.284,4.70,Water quality parameters within acceptable range.
9,Village_19,0.554,2.63,Water quality parameters within acceptable range.


In [26]:
summary_table = latest_data[["village_name", "population", "predicted_risk",
                               "water_quality_note", "recommendation"]]

summary_table = summary_table.sort_values(by="predicted_risk", ascending=False)
summary_table

,village_name,population,predicted_risk,water_quality_note,recommendation
0,Village_10,3873,Medium,Water quality parameters within acceptable range.,Risk detected; conduct manual water quality in...
5,Village_4,9026,Medium,Water quality parameters within acceptable range.,Risk detected; conduct manual water quality in...
8,Village_7,2385,Medium,Water quality parameters within acceptable range.,Improve sanitation infrastructure; monitor clo...
9,Village_19,5097,Medium,Water quality parameters within acceptable range.,Risk detected; conduct manual water quality in...
10,Village_8,4743,Medium,Turbidity above acceptable threshold (5 NTU).,Improve sanitation infrastructure; monitor clo...
13,Village_13,4643,Medium,Water quality parameters within acceptable range.,Improve sanitation infrastructure; monitor clo...
14,Village_14,8789,Medium,Water quality parameters within acceptable range.,Improve sanitation infrastructure; monitor clo...
18,Village_9,8355,Medium,Chlorine level below WHO-recommended minimum (...,Improve sanitation infrastructure; monitor clo...
1,Village_18,961,Low,Water quality parameters within acceptable range.,No immediate action needed; continue routine m...
4,Village_3,1575,Low,Water quality parameters within acceptable range.,No immediate action needed; continue routine m...


In [28]:
summary_table.to_csv("../data/processed/village_recommendations.csv", index=False)
print("Saved village-level recommendations.")

Saved village-level recommendations.
